# Adjacency Representations

```{contents}
:local:
:depth: 2
```

## Adjacency Matrices

```{index} adjacency matrix; vertex index; symmetry
```

An adjacency matrix reserves one cell for each ordered pair of vertices. A row describes outgoing connections from its vertex; a column describes incoming connections to its vertex. Map symbolic labels to dense integer indexes before accessing the array.

For the six-room undirected graph, a Boolean matrix has 36 cells, although only eight are true. Four logical edges appear twice, once in each direction. Its diagonal is false because the model rejects self-loops. An isolated vertex still has a row and column, both empty.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

string[] rooms = { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" };
var index = rooms.Select((room, i) => (room, i)).ToDictionary(x => x.room, x => x.i);
bool[,] connected = new bool[rooms.Length, rooms.Length];
foreach (var (a, b) in new[] { ("Lobby", "Hall"), ("Hall", "Lab"), ("Hall", "Office"), ("Office", "Storage") })
{
    connected[index[a], index[b]] = true;
    connected[index[b], index[a]] = true;
}
foreach (string room in rooms)
{
    int row = index[room];
    Console.WriteLine($"{room}: {string.Join(" ", Enumerable.Range(0, rooms.Length).Select(col => connected[row, col] ? 1 : 0))}");
}

The Hall row is `1 0 1 1 0 0`; the Remote row is all zeros. In an undirected matrix, `[i, j]` equals `[j, i]`. A directed matrix need not be symmetric. Do not infer a reverse arc merely because an undirected example wrote both cells.

Once indexes are known, direct edge lookup reads one cell in O(1) time. Enumerating a vertex's neighbors scans its entire row, Θ(V) time. Adding or removing vertices in a tightly sized matrix can require a new array and remapping many indexes. Keeping spare capacity or inactive indexes changes that tradeoff and needs its own policy.

## Weights and Absence

```{index} weighted adjacency matrix; nullable weight; zero-weight edge
```

A nullable weight matrix separates a missing edge (`null`) from an existing edge with weight zero. Initializing every cell to zero and treating zero as absence would erase valid zero-minute routes. The diagonal remains null under this simple-graph contract; a zero-cost path from a vertex to itself does not imply a stored self-loop.

In [ ]:
using System;
int?[,] minutes = new int?[3, 3];
minutes[0, 1] = minutes[1, 0] = 3;
minutes[1, 2] = minutes[2, 1] = 0;
Console.WriteLine($"edge 1-2 exists={minutes[1, 2].HasValue}, weight={minutes[1, 2]}");
Console.WriteLine($"edge 0-2 exists={minutes[0, 2].HasValue}");
Console.WriteLine($"stored loop 0-0={minutes[0, 0].HasValue}");

## Adjacency Collections

```{index} adjacency list; neighbor map; sparse graph
```

An adjacency list stores each vertex's actual neighbors. The term includes several concrete choices: a list, a set, or a dictionary of neighbor-to-weight pairs. These choices have different lookup and update costs.

A list supports neighbor iteration but must scan to find a particular neighbor, and it can contain duplicates unless insertion checks for them. A hash set supports expected constant-time membership for unweighted edges. A neighbor dictionary supports expected constant-time lookup and weight replacement while keeping one entry per neighbor. All retain explicit entries for isolated vertices.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var adjacency = new Dictionary<string, Dictionary<string, int>>(StringComparer.Ordinal);
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Remote" })
    adjacency.Add(room, new Dictionary<string, int>(StringComparer.Ordinal));
adjacency["Lobby"]["Hall"] = adjacency["Hall"]["Lobby"] = 4;
adjacency["Hall"]["Lab"] = adjacency["Lab"]["Hall"] = 2;
foreach (string room in adjacency.Keys.OrderBy(r => r, StringComparer.Ordinal))
    Console.WriteLine($"{room}: {string.Join(", ", adjacency[room].Keys.OrderBy(r => r, StringComparer.Ordinal))}");
Console.WriteLine($"Hall-Lab weight={adjacency["Hall"]["Lab"]}");

In this undirected model, each logical edge has two stored adjacency entries. The space bound is Θ(V + E), since doubling E changes only the constant factor. A directed outgoing adjacency representation stores one entry per arc. Forgetting the reverse entry in an undirected update creates an asymmetric representation of a supposedly symmetric relationship.

Sorting the keys above is a reporting choice, not a hash-table guarantee. The underlying neighbor dictionary does not promise numerical or alphabetical traversal order. Later graph algorithms may produce different valid visit orders when neighbor enumeration order differs.

## Density and Workload Costs

```{index} graph density; dense graph; representation cost
```

For a simple undirected graph with V vertices, the maximum number of edges is `V(V - 1) / 2`: choose an unordered pair of distinct endpoints. The directed maximum without loops is `V(V - 1)` because each pair has two possible directions. Density is actual edges divided by the relevant maximum. For fewer than two vertices, this chapter reports density zero to avoid division by zero.

| Operation | Matrix with known indexes | List per vertex | Hash neighbor map |
| --- | --- | --- | --- |
| Storage | Θ(V²) | Θ(V + E) | Θ(V + E) |
| Direct edge lookup | O(1) | O(deg(u)) | Expected O(1) |
| Iterate neighbors of u | Θ(V) | Θ(deg(u)) | Θ(deg(u)) |
| Set edge weight | O(1) | O(deg(u)) to find it | Expected amortized O(1) |
| Add an isolated vertex | May require Θ(V²) copying | Amortized O(1) in a growable indexed collection | Expected amortized O(1) |

Hash bounds assume well-distributed hashing and fixed-cost label operations; growth can make an individual insertion expensive. Label-to-index lookup is additional work for the matrix. The table's update costs describe directed entries; an undirected edge updates two entries with the same asymptotic bound.

In [ ]:
using System;
int vertices = 50, edges = 60;
long possible = (long)vertices * (vertices - 1) / 2;
double density = possible == 0 ? 0 : (double)edges / possible;
Console.WriteLine($"possible edges={possible}, density={density:P2}");
Console.WriteLine($"matrix cells={(long)vertices * vertices}, undirected adjacency entries={2 * edges}");

This graph has 60 of 1225 possible edges, about 4.90% density. A matrix reserves 2500 pair cells; adjacency collections store 120 neighbor entries plus 50 vertex entries. These are counts of storage locations, not equal-size bytes. Dictionary objects, hashing, and arrays have different overheads, so small graphs may favor a matrix even when sparse.

Choose by operations as well as density. Frequent pair checks with a fixed dense vertex set favor a matrix; scanning neighbors in a large sparse graph favors adjacency collections. No fixed density cutoff is a universal decision rule.

## A Protected Route Graph

```{index} graph API; undirected invariant; defensive copy
```

Encapsulation should protect graph rules, not merely hide a field's name. `RouteGraph` uses private neighbor dictionaries and returns copied arrays for vertices and neighbors. A caller can edit its copy without changing the stored graph. Returning a mutable collection as `IEnumerable` would not protect it: a caller could cast it back to the original collection type.

The contract requires explicit vertices. `SetEdge` checks both endpoints and weight rules before changing either direction. A new logical edge increases EdgeCount once; replacing its weight changes both reciprocal entries without changing EdgeCount. `TryGetWeight` returns false for an absent endpoint or edge, while degree/neighbor queries throw for an unknown vertex so it cannot masquerade as an isolate.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
Console.WriteLine($"vertices={graph.VertexCount}, edges={graph.EdgeCount}, valid={graph.IsValid()}");
Console.WriteLine($"replacement inserted={graph.SetEdge("Hall", "Lab", 5)}, edges={graph.EdgeCount}");
Console.WriteLine($"Hall-Lab found={graph.TryGetWeight("Hall", "Lab", out int minutes)}, minutes={minutes}");
var snapshot = graph.Neighbors("Hall");
snapshot[0] = new KeyValuePair<string, int>("Injected", 999);
Console.WriteLine($"snapshot changed graph={graph.TryGetWeight("Hall", "Injected", out _)}");
Console.WriteLine($"Remote degree={graph.Degree("Remote")}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

## Removal, Invariants, and Testing

```{index} edge removal; vertex removal; degree sum; graph validation
```

Removing an undirected edge deletes both reciprocal entries and reduces EdgeCount once. Removing a vertex deletes every reverse reference before removing its own dictionary. Its degree tells you how many logical edges disappear. Other vertices stay present even if they become isolated.

Trace removing Hall from the six-room example. Hall's neighbors are Lobby, Lab, and Office. Remove Hall from each of those neighbor maps, subtract three from EdgeCount, and delete Hall. Five vertices and one edge remain; Lobby and Lab are now isolates, Office–Storage remains, and Remote is unchanged. Removing Hall again returns false.

The representation invariant requires every neighbor label to name an existing vertex, no self-loops, nonnegative weights, equal reverse weights, and degree sum `2 * EdgeCount`. `IsValid` checks those rules in expected Θ(V + E) time. It still cannot prove that the data matches the intended floor plan, so compare edge membership and weights with an independent model too.

With neighbor hashes, `RemoveEdge` takes expected O(1) time. `RemoveVertex(u)` takes expected O(deg(u)) time because it removes a reference at each neighbor. The code does not mutate u's neighbor map while enumerating it; it changes the distinct maps belonging to those neighbors. Self-loops are rejected, so a reverse map cannot be the same map being enumerated.

`Degree` reads a stored dictionary count in expected O(1). `Neighbors` copies deg(u) entries, so it takes Θ(deg(u)) time and space. `Vertices` copies V labels. Defensive copying has a cost, but keeps callers from bypassing update rules.

## Exercise

```{index} undirected edge removal; exercise
```

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// Exercise: remove both stored directions of one undirected edge.
var adjacency = new Dictionary<string, Dictionary<string, int>>
{
    ["A"] = new() { ["B"] = 3 },
    ["B"] = new() { ["A"] = 3, ["C"] = 0 },
    ["C"] = new() { ["B"] = 0 }
};
// Your code starts here.
/* TODO: remove A-B in both directions while preserving B-C. */
// Your code ends here.
Console.WriteLine($"A-B={adjacency["A"].ContainsKey("B")}, B-A={adjacency["B"].ContainsKey("A")}");
Console.WriteLine($"B-C={adjacency["B"]["C"]}, C-B={adjacency["C"]["B"]}");

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// Solution
var adjacency = new Dictionary<string, Dictionary<string, int>>
{
    ["A"] = new() { ["B"] = 3 },
    ["B"] = new() { ["A"] = 3, ["C"] = 0 },
    ["C"] = new() { ["B"] = 0 }
};
adjacency["A"].Remove("B"); adjacency["B"].Remove("A");
Console.WriteLine($"A-B={adjacency["A"].ContainsKey("B")}, B-A={adjacency["B"].ContainsKey("A")}");
Console.WriteLine($"B-C={adjacency["B"]["C"]}, C-B={adjacency["C"]["B"]}");

A-B=False, B-A=False
B-C=0, C-B=0


```{rubric} Footnotes
```

[^1]: Directed graphs may also maintain incoming adjacency collections. That uses additional storage but makes removal of incoming arcs and reverse-neighbor queries faster.
[^2]: Representation costs depend on the concrete neighbor collection; an adjacency list implemented with hash maps differs from one implemented with ordinary lists.